# CMSC 173 &middot; Machine Learning &mdash; Week 2 Lab
## Parameter Estimation: Method of Moments vs Maximum Likelihood

Last week's lecture asked a single question: given data, how do you recover the
parameters of the distribution that produced it? You saw two answers &mdash; **Method of
Moments (MoM)** and **Maximum Likelihood (MLE)**. This lab makes both concrete on data
you generate yourself, so you can see where they agree, where they differ, and why MLE
is the one the rest of this course leans on.

**Not graded.** About 50 minutes. NumPy only &mdash; still no scikit-learn (that arrives in
week 4, after you have fit linear regression by hand in week 3).

Work top to bottom. Every *Answer here* is a place to write, not decoration &mdash; they are
how I tell what landed.

---
## Part 0 &middot; Setup

Run this. Same seed as week 1, so your "random" numbers match mine.

In [ ]:
import sys
import numpy as np

print("Python", sys.version.split()[0])
print("NumPy ", np.__version__)
rng = np.random.default_rng(173)
print("\nReady.")

---
## Part 1 &middot; Moments are just averages

A **moment** is an average of a power of the data. The first raw moment is the mean; the
variance is the second *central* moment. Method of Moments rests on one idea: the sample
versions should match the theoretical ones.

- raw $k$-th moment:  $m_k = \frac{1}{n}\sum_i x_i^k$
- mean $= m_1$;  variance $= \frac{1}{n}\sum_i (x_i - \bar{x})^2$ (second *central* moment)

Run the cell, then answer.

In [ ]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

m1  = data.mean()                     # first raw moment
m2  = (data**2).mean()                # second raw moment
var = ((data - m1)**2).mean()         # second central moment, divided by n

print("m1 (mean)           =", m1)
print("m2 (raw 2nd moment) =", m2)
print("variance (/n)       =", round(var, 4))
print("m2 - m1**2          =", round(m2 - m1**2, 4))   # compare to the variance

**Answer here** (double-click to edit):

1. `m2 - m1**2` equals the variance printed above. In one line of algebra, show why the
   raw second moment minus the mean-squared gives the central variance.
   &rarr; *your answer*

2. `var` divides by $n$. NumPy's `data.var(ddof=1)` divides by $n-1$. Compute both.
   Which is larger, and why does the gap shrink as $n$ grows?
   &rarr; *your answer*

---
## Part 2 &middot; Method of Moments, on a Normal

Now run it the other way: start from data whose *true* parameters you know, and see how
close the estimates land. For a Normal, MoM sets $\hat{\mu} = \bar{x}$ and
$\hat{\sigma}^2 = \frac{1}{n}\sum (x_i-\bar{x})^2$. Because we generate the data, we can
grade the estimator against the truth.

In [ ]:
true_mu, true_sigma = 5.0, 2.0
sample = rng.normal(true_mu, true_sigma, size=200)

def mom_normal(x):
    """Method-of-Moments estimates (mu_hat, var_hat) for a Normal. NumPy only."""
    mu_hat  = x.mean()
    var_hat = ((x - mu_hat)**2).mean()      # /n : the MoM (and, as we'll see, MLE) variance
    return mu_hat, var_hat

mu_hat, var_hat = mom_normal(sample)
print(f"true:     mu = {true_mu}, sigma^2 = {true_sigma**2:.2f}")
print(f"MoM est.: mu = {mu_hat:.3f}, sigma^2 = {var_hat:.3f}")

**Answer here:**

1. Change `size=200` to `size=20`, run, then change it back. Do the estimates get better
   or worse? State the general rule in one sentence.
   &rarr; *your answer*

2. MoM produced $\hat{\sigma}^2$ by matching the second moment. For the Normal it is always
   valid, but name one thing that can make a MoM estimate come out *invalid* (hint: recall
   the Gamma example from lecture).
   &rarr; *your answer*

---
## Part 3 &middot; Maximum Likelihood

MLE asks a different question: which parameters make the data you *actually observed* most
probable? You maximise the log-likelihood

$$\ell(\mu, \sigma^2) = \sum_{i=1}^{n} \log f(x_i \mid \mu, \sigma^2).$$

For the Normal this has a closed form, and here is the punchline: it lands on the **same**
$\hat{\mu}$ and $\hat{\sigma}^2$ as MoM. Let's confirm that numerically rather than take it
on faith.

In [ ]:
def normal_loglik(x, mu, var):
    """Total log-likelihood of x under N(mu, var). Written out, no scipy."""
    n = len(x)
    return -0.5 * n * np.log(2 * np.pi * var) - ((x - mu)**2).sum() / (2 * var)

# Evaluate the log-likelihood across a grid of mu (var fixed at the MLE value)
mus = np.linspace(4.0, 6.0, 401)
lls = np.array([normal_loglik(sample, m, var_hat) for m in mus])
mu_grid_best = mus[lls.argmax()]

print(f"MLE (closed form) mu_hat = {sample.mean():.3f}")
print(f"grid-search best  mu     = {mu_grid_best:.3f}")
print(f"MoM               mu     = {mu_hat:.3f}   (all three agree)")

**Answer here:**

1. We maximised the **log**-likelihood, not the likelihood. Give the two practical reasons
   from lecture &mdash; one about the arithmetic (very small numbers), one about the calculus.
   &rarr; *your answer*

2. Grid search and closed form agree here. Name one model from later in this course that has
   **no** closed form, so you would *have* to search numerically for the MLE.
   &rarr; *your answer*

---
## Part 4 &middot; Is the estimator any good?

An estimator is a recipe; run it on different samples and you get different answers. Two
questions from lecture: is it **biased** (wrong on average), and is it **consistent**
(does it converge as $n \to \infty$)? We can just *simulate* the answer: draw many samples,
estimate each, and look at the spread.

In [ ]:
true_var = true_sigma**2
sizes  = [10, 50, 500]
trials = 2000

print(f"true sigma^2 = {true_var:.2f}\n")
print(f"{'n':>5} {'mean of /n est':>16} {'mean of /(n-1) est':>20}")
for n in sizes:
    v_biased, v_unbiased = [], []
    for _ in range(trials):
        s = rng.normal(true_mu, true_sigma, size=n)
        v_biased.append(((s - s.mean())**2).mean())   # /n     (the MLE)
        v_unbiased.append(s.var(ddof=1))              # /(n-1)  (unbiased)
    print(f"{n:>5} {np.mean(v_biased):>16.3f} {np.mean(v_unbiased):>20.3f}")

**Answer here:**

1. Across many trials the `/n` estimate averages *below* the true $\sigma^2$; the `/(n-1)`
   one sits on it. That gap is the **bias** of the MLE variance. Read its rough size off the
   `n = 10` row (about what fraction of the truth?).
   &rarr; *your answer*

2. Both columns approach the truth as $n$ grows &mdash; that is **consistency**, and it is why
   the MLE's bias is usually forgiven. In one sentence: when would you still insist on the
   unbiased `/(n-1)` version?
   &rarr; *your answer*

---
## Part 5 &middot; A distribution where MoM and MLE fully coincide

For a **Poisson**($\lambda$) &mdash; counts of independent events, like dengue cases per
barangay per week &mdash; both methods give the same estimate: $\hat{\lambda} = \bar{x}$.
Confirm it, and notice the whole machinery collapses to "take the average" when the
distribution is simple enough.

In [ ]:
true_lambda = 3.5
counts = rng.poisson(true_lambda, size=300)

lam_mom = counts.mean()   # MoM:  E[X] = lambda   ->  lambda_hat = xbar
lam_mle = counts.mean()   # MLE:  d/dlam[(sum x) log lam - n lam] = 0  ->  lambda_hat = xbar

print(f"true lambda = {true_lambda}")
print(f"MoM  lambda = {lam_mom:.3f}")
print(f"MLE  lambda = {lam_mle:.3f}")

**Answer here:**

1. For the Poisson, MoM and MLE are identical. From lecture, name one distribution where
   they are **not**, and which of the two you would trust more there.
   &rarr; *your answer*

2. `rng.poisson` only makes sense for non-negative integer counts. Give one real Philippine
   dataset that is genuinely count-like (Poisson-shaped), and one that *looks* count-like but
   would mislead a Poisson model.
   &rarr; *your answer*

---
## Part 6 &middot; Where you actually are

Same as last week &mdash; set the pace honestly.

Replace each `-` with one of: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Sample mean and variance in NumPy | - |
| Raw vs central moments | - |
| Writing a log-likelihood | - |
| Bias vs consistency of an estimator | - |
| Reading a small simulation table | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one sentence: why does this course prefer MLE over MoM as its default?**
&rarr; *your answer*

---
## Stretch &mdash; optional

The required part is done; nothing below is graded.

### Stretch 1 &middot; How sure are you? (bootstrap)

You reported a single $\hat{\lambda}$. How much would it wobble on a different sample? The
**bootstrap**: resample your data *with replacement* many times, re-estimate each time, and
look at the spread. This is given &mdash; read it and run it.

In [ ]:
B = 2000
boot = np.empty(B)
for i in range(B):
    resample = rng.choice(counts, size=len(counts), replace=True)
    boot[i]  = resample.mean()          # re-estimate lambda on each resample

print(f"lambda_hat           = {counts.mean():.3f}")
print(f"bootstrap std. error = {boot.std():.3f}")
print(f"~95% interval        = [{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]")

### Stretch 2 &middot; Exponential, from scratch

Waiting times between events are often **Exponential**($\lambda$): $f(x) = \lambda e^{-\lambda x}$,
and the MLE is $\hat{\lambda} = 1/\bar{x}$. Data is below (NumPy parameterises the exponential
by its *mean*, which is $1/\lambda$). Write the estimator and compare to the truth.

In [ ]:
true_rate = 0.5
waits = rng.exponential(1 / true_rate, size=250)   # mean = 1/rate

# your code here: estimate the rate (lambda) by MLE, then print truth vs estimate


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type, so
the token never gets saved inside your notebook.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 2

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/2/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 2 submission page](https://portal.latarak.com/course/cmsc173/lab/2/submit) and upload it.

Blank cells are fine and guesses are fine. What is not useful is polishing this until it hides
what you actually knew &mdash; that just moves the surprise to a later week, where it costs more.